<a href="https://colab.research.google.com/github/kyurimyang/AIcapston1/blob/main/train/colab_stage2_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 유담 인식 모델 ② 파인튜닝 / ③ 증강 (Colab T4)
- 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 위에서부터 실행
- Drive의 `yudam/yudam_colab.zip`(data/train, data/val, train/ 코드·manifest)만 사용한다. **test·gallery는 올리지 않는다.**
- AI Hub 약관: 데이터 원본·가공본 공유 금지 → Drive 폴더 링크 공유하지 말 것

In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음 - 런타임 유형을 확인하세요')

True Tesla T4


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Drive의 압축 파일을 Colab 로컬 디스크로 풀기 (Drive에서 바로 읽으면 느리다)
!mkdir -p /content/yudam && cd /content/yudam && unzip -q -o /content/drive/MyDrive/yudam/yudam_colab.zip
!ls /content/yudam /content/yudam/data
!pip -q install open_clip_torch albumentations

/content/yudam:
data  train

/content/yudam/data:
train  val


In [7]:
# 동작 점검: 아주 짧게 (3스텝). 에러 없이 끝나면 아래 본 학습으로
%cd /content/yudam
!python train/train_metric.py --stage 2 --manifest train/manifest.csv --data-root data --out-dir /content/out_check --steps 3

/content/yudam
stage2 device=cuda train 유물 141점 / val 유물 35점

open_clip_model.safetensors: downloading bytes:  38% 229M/599M [00:01<00:01, 226MB/s, 20.4MB/s  ]
open_clip_model.safetensors: downloading bytes:  66% 398M/599M [00:02<00:00, 268MB/s, 33.0MB/s  ]
open_clip_model.safetensors: downloading bytes: 100% 428M/428M [00:05<00:00, 79.2MB/s, 37.7MB/s  ]
open_clip_model.safetensors: reconstructing file: 100% 599M/599M [00:05<00:00, 111MB/s, 47.5MB/s  ]
학습 파라미터 28.7M (마지막 4블록 + projection)
val Top-1 (학습 전): 0.864
step 3/3 loss 2.4376
val Top-1 (학습 후): 0.914
저장: /content/out_check/ckpt_stage2.pt


## ② 파인튜닝 (증강 없음)

In [8]:
%cd /content/yudam
!python train/train_metric.py --stage 2 --manifest train/manifest.csv --data-root data --out-dir /content/out --steps 300
# 학습 전후 'val Top-1' 두 줄을 메모해 두기

/content/yudam
stage2 device=cuda train 유물 141점 / val 유물 35점
학습 파라미터 28.7M (마지막 4블록 + projection)
val Top-1 (학습 전): 0.864
step 20/300 loss 1.5431
step 40/300 loss 1.2780
step 60/300 loss 1.1759
step 80/300 loss 1.1530
step 100/300 loss 1.1533
step 120/300 loss 1.1472
step 140/300 loss 1.1275
step 160/300 loss 1.1180
step 180/300 loss 1.1212
step 200/300 loss 1.1095
step 220/300 loss 1.1111
step 240/300 loss 1.1099
step 260/300 loss 1.1078
step 280/300 loss 1.1061
step 300/300 loss 1.1100
val Top-1 (학습 후): 0.979
저장: /content/out/ckpt_stage2.pt


In [9]:
!mkdir -p /content/drive/MyDrive/yudam/out && cp /content/out/ckpt_stage2.pt /content/drive/MyDrive/yudam/out/
!ls -la /content/drive/MyDrive/yudam/out

total 584560
drwx------ 2 root root      4096 Nov 11 10:08 .
drwx------ 3 root root      4096 Nov 11 10:08 ..
-rw------- 1 root root 598580531 Nov 11 10:08 ckpt_stage2.pt


## ③ + 증강 (②와 같은 설정, 증강만 추가). ② 결과를 확인한 뒤 실행

In [10]:
%cd /content/yudam
!python train/train_metric.py --stage 3 --manifest train/manifest.csv --data-root data --out-dir /content/out --steps 300
!cp /content/out/ckpt_stage3.pt /content/drive/MyDrive/yudam/out/

/content/yudam
stage3 device=cuda train 유물 141점 / val 유물 35점
학습 파라미터 28.7M (마지막 4블록 + projection)
val Top-1 (학습 전): 0.864
step 20/300 loss 2.1777
step 40/300 loss 2.0332
step 60/300 loss 1.6804
step 80/300 loss 1.5151
step 100/300 loss 1.7420
step 120/300 loss 1.4898
step 140/300 loss 1.4243
step 160/300 loss 1.3610
step 180/300 loss 1.4041
step 200/300 loss 1.3362
step 220/300 loss 1.3129
step 240/300 loss 1.2873
step 260/300 loss 1.2653
step 280/300 loss 1.3464
step 300/300 loss 1.3544
val Top-1 (학습 후): 0.979
저장: /content/out/ckpt_stage3.pt


## 끝난 뒤
Drive `yudam/out/`의 `ckpt_stage2.pt`, `ckpt_stage3.pt`를 PC의 `models/`에 넣고, 학습 전후 val Top-1 숫자를 알려 주면 인덱스 재구축·test 평가를 이어서 한다.